# Decision layer

Ex post oracle: для каждого origin и горизонта выбираем текущую ставку из $[0,10]$, минимизирующую $(π_{fact}-π_{forecast})^2$. Известные на дату origin лаги ставки остаются неизменными.

In [7]:
import json
from pathlib import Path
stack_nb = json.loads(Path('stacking.ipynb').read_text())
exec(next(''.join(c['source']) for c in stack_nb['cells'] if c.get('id') == 'stacking-setup'))
del stack_nb

RATE_GRID = np.linspace(0, 10, 401)
weights = {h: cloudpickle.load(open(STACKING_DIR / f'stacking_weights_h{h}.pkl', 'rb')) for h in range(1, 9)}
active = {}
for h, values in weights.items():
    kept = {k: v for k, v in values.items() if v > 1e-8}; total = sum(kept.values())
    active[h] = {k: v / total for k, v in kept.items()}
pysr_models = {(h, r): cloudpickle.load(open(PYSR_MODEL_DIR / f'pysr_cpiaucsl_h{h}_{r}_fedfunds.pkl', 'rb'))
               for h in range(1, 9) for r in {k.split('|')[1] for k in active[h]}}
infl_models = {(h, k): load_model(*k.split('|'), h) for h in range(1, 9) for k in active[h]}

22 residual-моделей × 22 моделей инфляции = 484; lambda=0.01


In [8]:
def ensemble_curve(origin, h, rates=RATE_GRID):
    result, X_by_rate = np.zeros(len(rates)), globals()[f'X_full_h{h}']
    for rate_name in {k.split('|')[1] for k in active[h]}:
        gap = rate_residuals['full'][h, rate_name]; base = pysr_inputs(data_full, gap).loc[[origin]]
        Z = pd.DataFrame(np.repeat(base.to_numpy(), len(rates), axis=0), columns=base.columns)
        expected = levels['full'].loc[origin] - gap.loc[origin]
        Z['FEDFUNDS_GAP_t'] = rates - expected
        phi = pysr_models[h, rate_name].predict(Z); X0 = X_by_rate[rate_name].loc[[origin]]
        X = pd.DataFrame(np.repeat(X0.to_numpy(), len(rates), axis=0), columns=X0.columns); X['PYSR_FEATURE'] = phi
        for key, weight in active[h].items():
            if key.split('|')[1] == rate_name: result += weight * infl_models[h, key].predict(X)
    return result

def optimal_rate(origin, h, target):
    forecast = ensemble_curve(origin, h)
    if np.ptp(forecast) < 1e-10: return np.nan, forecast[0], False
    loss = (target - forecast) ** 2; best = np.flatnonzero(np.isclose(loss, loss.min(), atol=1e-12, rtol=0))
    i = best[len(best) // 2]
    return RATE_GRID[i], forecast[i], True

In [9]:
rows = []
for origin in data_full.index[data_full.index >= train_end]:
    pos = data_full.index.get_loc(origin)
    for h in range(1, min(8, len(data_full) - pos - 1) + 1):
        date = data_full.index[pos + h]; target = data_full.loc[date, 'CPIAUCSL']
        rate, prediction, identified = optimal_rate(origin, h, target)
        rows.append({'origin': origin, 'Date': date, 'horizon': h, 'optimal_rate': rate,
                     'rate_identified': identified, 'inflation_pred': prediction, 'inflation_true': target})

decision_oracle = pd.DataFrame(rows)
decision_oracle['inflation_sq_error'] = (decision_oracle.inflation_pred - decision_oracle.inflation_true) ** 2
oracle_mse = decision_oracle.groupby('horizon').agg(MSE=('inflation_sq_error', 'mean'), n=('origin', 'size'),
                                                       identified_rates=('rate_identified', 'sum'))
decision_oracle.to_csv(STACKING_DIR / 'decision_layer_oracle.csv', index=False)
oracle_mse.to_csv(STACKING_DIR / 'decision_layer_oracle_mse.csv')
print('Overall oracle MSE:', decision_oracle.inflation_sq_error.mean())
oracle_mse.round(8)

Overall oracle MSE: 2.2816596457603746e-05


,MSE,n,identified_rates
horizon,,,
1,0.000011,9,9
2,0.000016,8,8
3,0.000033,7,7
4,0.000039,6,6
5,0.000010,5,0
6,0.000036,4,4
7,0.000024,3,3
8,0.000023,2,2
